# 用 Python 实现 BPE（Byte Pair Encoding）

BPE 的核心流程如下：
1. 把每个词拆成字符的序列，并添加词尾标记 `</w>`
2. 统计所有相邻 token pair 的频率
3. 找到频率最高的 pair
4. 把这个 pair 合并成一个新的 token
5. 重复多轮，得到 merge rules
6. 用 merge rules 对新词进行编码

例如：
```text
low      -> l o w </w>
lower    -> l o w e r</w>
lowest   -> l o w e s t</w>
```
如果高频对为 `lo`，于是，在此规则下，上面的词将会被分词为
```text
low      -> lo w </w>
lower    -> lo w e r</w>
lowest   -> lo w e s t</w>
```

## 一、加载语料

本文使用的是 `name.txt`, 它的内容是：美国 SSA（Social Security Administration）2018 年最常见的约 32K 个英文名字 / 婴儿名

In [1]:
words = open('./resources/names.txt', 'r').read().splitlines()

In [2]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

## 二、初始化词表
把每个词拆成字符的序列，并添加词尾标记 `</w>`

In [3]:
def init_vocab(corpus):
    """
    示例：
    "low" -> ("l", "o", "w", "</w>")

    """
    vocab_ls = []
    for word in corpus:
        vocab_ls.append(tuple(list(word) + ["</w>"]))

    return vocab_ls

## 三、频率统计

In [4]:
def get_pair_counts(vocab):
    """
    统计所有相邻 token pair 的频率

    示例：
    ("l", "o", "w", "</w>")
    ("l", "o", "w", "</w>")
    -> ("l", "o"), ("o", "w"), ("w", "</w>")

    """
    pair_counts = {}
    for item in vocab:
        pairs = tuple(zip(item, item[1:]))
        for pair in pairs:
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

    return pair_counts

In [5]:
pair_counts = get_pair_counts([("l", "o", "w", "</w>"), ("a", "o", "w", "</w>")])
pair_counts

{('l', 'o'): 1, ('o', 'w'): 2, ('w', '</w>'): 2, ('a', 'o'): 1}

In [6]:
names_pair_counts = get_pair_counts(words)
sorted(names_pair_counts.items(), key=lambda x: x[1], reverse=True)[:10]

[(('a', 'n'), 5438),
 (('a', 'r'), 3264),
 (('e', 'l'), 3248),
 (('r', 'i'), 3033),
 (('n', 'a'), 2977),
 (('l', 'e'), 2921),
 (('e', 'n'), 2675),
 (('l', 'a'), 2623),
 (('m', 'a'), 2590),
 (('a', 'l'), 2528)]

In [7]:
def get_best_pair(pair_counts):
    """
    选择频率最高的 pair
    """
    if not pair_counts:
        return None

    best_pair, counts = sorted(pair_counts.items(), key=lambda v: v[1], reverse=True)[0]
    return best_pair, counts

In [8]:
get_best_pair(pair_counts)

(('o', 'w'), 2)

## 四、合并频率最高的 token pair

In [9]:
def merge_pair_in_tokens(tokens, pair):
    """
    在一个 token序列里合并指定的 pair

    示例：
    tokens = ("l", "o", "w", "</w>")
    pair = ("l", "o")
    结果 = ("lo", "w", "</w>")

    """
    merge_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merge_tokens.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            merge_tokens.append(tokens[i])
            i += 1
    return tuple(merge_tokens)


In [10]:
merge_pair_in_tokens(("l", "o", "w", "</w>"), ("l", "o"))

('lo', 'w', '</w>')

In [11]:
def merge_vocab(vocab, pair):
    """
    对整个词表应用一次 pair 合并
    """
    merged_vocab_ls = []
    for item in vocab:
        merged_vocab_ls.append(merge_pair_in_tokens(item, pair))

    return merged_vocab_ls


In [12]:
merge_vocab([("l", "o", "w", "</w>"), ("a", "o", "w", "</w>"), ("l", "o", "v", "</w>")], ("l", "o"))

[('lo', 'w', '</w>'), ('a', 'o', 'w', '</w>'), ('lo', 'v', '</w>')]

## 五、训练

In [13]:
def train_bpe(corpus, num_merges=10, verbose=True):
    """
    训练 BPE

    目标：
    - 初始化词表
    - 重复统计 pair
    - 选择最高频 pair
    - 合并词表
    - 保存每一步的 merge rule
    """
    merges = []
    vocab_ls = init_vocab(corpus)

    if verbose:
        print("=" * 60)
        print("BPE 训练开始")
        print("=" * 60)
        print(f"语料数量：{len(corpus)}")
        print(f"计划合并次数：{num_merges}")
        print()
        print("初始化词表：")
        for idx, tokens in enumerate(vocab_ls, start=1):
            print(f"{idx:>2}. {tokens}")
        print("=" * 60)

    for step in range(num_merges):
        pair_counts = get_pair_counts(vocab_ls)
        best_pair, count = get_best_pair(pair_counts)

        if best_pair is None:
            if verbose:
                print()
                print("没有可继续合并的 pair，训练提前结束。")
            break

        merge_token = best_pair[0] + best_pair[1]
        vocab_ls = merge_vocab(vocab_ls, best_pair)
        merges.append(best_pair)

        if verbose:
            print()
            print(f"Step {step + 1}/{num_merges}")
            print("-" * 60)
            print(f"最高频 Pair ：{best_pair}")
            print(f"出现次数    ：{count}")
            print(f"合并规则    ：{best_pair[0]} + {best_pair[1]} -> {merge_token}")
            print()
            print("合并后的词表：")
            for idx, tokens in enumerate(vocab_ls, start=1):
                print(f"{idx:>2}. {tokens}")

    if verbose:
        print()
        print("=" * 60)
        print("BPE 训练完成")
        print("=" * 60)
        print("最终 Merge Rules：")
        for idx, pair in enumerate(merges, start=1):
            print(f"{idx:>2}. {pair[0]} + {pair[1]} -> {pair[0] + pair[1]}")

        print()
        print("最终词表：")
        for idx, tokens in enumerate(vocab_ls, start=1):
            print(f"{idx:>2}. {tokens}")

    return merges, vocab_ls

### 训练语料的 demo

In [14]:
corpus = [
        "low",
        "lower",
        "lowest",
        "newer",
        "wider"
    ]

merge_rules, vocab_ls = train_bpe(corpus)

BPE 训练开始
语料数量：5
计划合并次数：10

初始化词表：
 1. ('l', 'o', 'w', '</w>')
 2. ('l', 'o', 'w', 'e', 'r', '</w>')
 3. ('l', 'o', 'w', 'e', 's', 't', '</w>')
 4. ('n', 'e', 'w', 'e', 'r', '</w>')
 5. ('w', 'i', 'd', 'e', 'r', '</w>')

Step 1/10
------------------------------------------------------------
最高频 Pair ：('l', 'o')
出现次数    ：3
合并规则    ：l + o -> lo

合并后的词表：
 1. ('lo', 'w', '</w>')
 2. ('lo', 'w', 'e', 'r', '</w>')
 3. ('lo', 'w', 'e', 's', 't', '</w>')
 4. ('n', 'e', 'w', 'e', 'r', '</w>')
 5. ('w', 'i', 'd', 'e', 'r', '</w>')

Step 2/10
------------------------------------------------------------
最高频 Pair ：('lo', 'w')
出现次数    ：3
合并规则    ：lo + w -> low

合并后的词表：
 1. ('low', '</w>')
 2. ('low', 'e', 'r', '</w>')
 3. ('low', 'e', 's', 't', '</w>')
 4. ('n', 'e', 'w', 'e', 'r', '</w>')
 5. ('w', 'i', 'd', 'e', 'r', '</w>')

Step 3/10
------------------------------------------------------------
最高频 Pair ：('e', 'r')
出现次数    ：3
合并规则    ：e + r -> er

合并后的词表：
 1. ('low', '</w>')
 2. ('low', 'er', '</w

合并的规则如下

In [15]:
merge_rules

[('l', 'o'),
 ('lo', 'w'),
 ('e', 'r'),
 ('er', '</w>'),
 ('low', '</w>'),
 ('low', 'er</w>'),
 ('low', 'e'),
 ('lowe', 's'),
 ('lowes', 't'),
 ('lowest', '</w>')]

最终的词表

In [16]:
vocab_ls

[('low</w>',),
 ('lower</w>',),
 ('lowest</w>',),
 ('n', 'e', 'w', 'er</w>'),
 ('w', 'i', 'd', 'er</w>')]

### 实际的语料数据集

In [17]:
names_merge_rules, names_vocab_ls = train_bpe(words, num_merges=100, verbose=False)

合并的规则如下（仅展示前 20 项）

In [18]:
len(names_merge_rules), names_merge_rules[:20]

(100,
 [('n', '</w>'),
  ('a', '</w>'),
  ('e', '</w>'),
  ('a', 'n'),
  ('a', 'r'),
  ('e', 'l'),
  ('a', 'l'),
  ('i', '</w>'),
  ('h', '</w>'),
  ('a', 'y'),
  ('e', 'r'),
  ('y', '</w>'),
  ('a', 'h</w>'),
  ('l', 'e'),
  ('a', 'm'),
  ('r', 'i'),
  ('a', 'n</w>'),
  ('o', 'n</w>'),
  ('a', 'i'),
  ('e', 'n')])

最终分词的词表如下（仅展示前 20 项）

In [19]:
len(names_vocab_ls), names_vocab_ls[:20]

(32033,
 [('em', 'm', 'a</w>'),
  ('o', 'li', 'v', 'ia</w>'),
  ('av', 'a</w>'),
  ('is', 'ab', 'ell', 'a</w>'),
  ('s', 'o', 'p', 'h', 'ia</w>'),
  ('ch', 'ar', 'l', 'o', 't', 't', 'e</w>'),
  ('mi', 'a</w>'),
  ('am', 'el', 'ia</w>'),
  ('h', 'ar', 'p', 'er</w>'),
  ('ev', 'el', 'yn</w>'),
  ('ab', 'ig', 'ai', 'l', '</w>'),
  ('e', 'mi', 'l', 'y</w>'),
  ('eli', 'z', 'ab', 'et', 'h</w>'),
  ('mi', 'l', 'a</w>'),
  ('ell', 'a</w>'),
  ('av', 'er', 'y</w>'),
  ('s', 'o', 'f', 'ia</w>'),
  ('c', 'am', 'il', 'a</w>'),
  ('ari', 'a</w>'),
  ('s', 'c', 'ar', 'le', 't', 't</w>')])

## 六、用训练好的 BPE merge rules 对新词分词

In [20]:
def encode_word(word, merges):
    """
    用训练好的 BPE merge rules 对新词分词
    """
    tokens = tuple(list(word) + ["</w>"])

    for pair in merges:
        tokens = merge_pair_in_tokens(tokens, pair)

    cleaned_tokens = []

    for token in tokens:
        if token == "</w>":
            continue
        cleaned_tokens.append(token.replace("</w>", ""))

    return tuple(cleaned_tokens)

In [22]:
encode_word("jordan", names_merge_rules)

('j', 'or', 'd', 'an')